# Examen de Recuperación — Servicio RAG para Information Retrieval

**ICCD753 Recuperación de Información 2026-A · Prof. Iván Carrera · EPN-FIS**

URL del servicio desplegado: **https://prone-snowiness-reviving.ngrok-free.dev**

Ejemplo de Uso: https://github.com/Alexandor31/ExamenRecuperacion/blob/main/DELIVERABLES.md

---

Este notebook documenta el desarrollo completo del servicio web RAG exigido por el examen de recuperación. Cada celda ejecuta una parte del pipeline; los artefactos persistentes (índice Chroma, *manifest*) se almacenan en `data/` para que el servicio pueda arrancar en frío sin reindexar.

## Índice

A. Configuración del entorno y dependencias
B. Adquisición y registro bibliográfico del corpus
C. Procesamiento del corpus (extracción, limpieza, *chunking*)
D. Indexación en base vectorial
E. *Retrieval* denso + re-ranking con cross-encoder
F. Generación de respuestas fundamentadas
G. Servicio web (FastAPI)
H. Despliegue y ejemplos de consumo
I. Códigos de estado HTTP
J. Limitaciones y decisiones de diseño

## A. Configuración del entorno

Instala las dependencias en el entorno actual. Si ejecutas el notebook en Hugging Face Spaces, el `Dockerfile` ya las instala; este paso es para ejecuciones locales.

In [1]:
%pip install -q -r requirements.txt

error: externally-managed-environment

× This environment is externally managed
╰─> To install Python packages system-wide, try apt install
    python3-xyz, where xyz is the package you are trying to
    install.
    
    If you wish to install a non-Debian-packaged Python package,
    create a virtual environment using python3 -m venv path/to/venv.
    Then use path/to/venv/bin/python and path/to/venv/bin/pip. Make
    sure you have python3-full installed.
    
    If you wish to install a non-Debian packaged Python application,
    it may be easiest to use pipx install xyz, which will manage a
    virtual environment for you. Make sure you have pipx installed.
    
    See /usr/share/doc/python3.12/README.venv for more information.

note: If you believe this is a mistake, please contact your Python installation or OS distribution provider. You can override this, at the risk of breaking your Python installation or OS, by passing --break-system-packages.
hint: See PEP 668 for the detai

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path('.').resolve()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
os.environ.setdefault('DATA_DIR', str(PROJECT_ROOT / 'data'))
os.environ.setdefault('CHROMA_DIR', str(PROJECT_ROOT / 'data' / 'chroma'))
os.environ.setdefault('CHROMA_COLLECTION', 'ir_corpus')
os.environ.setdefault('LOG_LEVEL', 'INFO')

# En HF Spaces, define LLM_API_KEY como secreto. En local, puedes usar:
# os.environ['LLM_API_KEY'] = 'gsk_...'

from ir_rag.config import Settings
settings = Settings.from_env()
print('Configuración cargada:')
print(f'  corpus_dir     = {settings.corpus_dir}')
print(f'  chroma_dir     = {settings.chroma_dir}')
print(f'  embedding      = {settings.embedding_model}')
print(f'  reranker       = {settings.reranker_model}')
print(f'  LLM model      = {settings.llm_model}')
print(f'  LLM key set?   = {bool(settings.llm_api_key)}')

Configuración cargada:
  corpus_dir     = /home/alexander/RI/ExamenSupletorio/corpus
  chroma_dir     = /home/alexander/RI/ExamenSupletorio/data/chroma
  embedding      = sentence-transformers/all-MiniLM-L6-v2
  reranker       = cross-encoder/ms-marco-MiniLM-L-6-v2
  LLM model      = llama-3.3-70b-versatile
  LLM key set?   = False


## B. Adquisición del corpus

### B.1 Bibliografía obligatoria

1. **Baeza-Yates & Ribeiro-Neto — *Modern Information Retrieval*** (1999). **Protegido por derechos de autor**: descárgalo de tu biblioteca o copia personal y colócalo en `corpus/baeza-yates-modern-ir.pdf`. Si la edición tiene metadatos distintos, ajusta `CORPUS_REGISTRY` en `src/ir_rag/corpus.py`.
2. **Manning, Raghavan & Schütze — *Introduction to Information Retrieval*** (2009). Acceso abierto desde Stanford NLP.

In [3]:
# Descarga los PDFs de acceso abierto. Si ya están en corpus/, se omiten.
!python3 scripts/download_corpus.py

  ✓ manning-introduction-ir.pdf already present (6,903,344 bytes)
  ✓ jurafsky-slp3.pdf already present (26,357,909 bytes)
  ✓ karpukhin-dpr-2020.pdf already present (383,508 bytes)
  ✓ nogueira-monobert-2019.pdf already present (179,659 bytes)
  ✓ robertson-bm25-perspective.pdf already present (423,997 bytes)

Reminder:
  Place your copy of Baeza-Yates & Ribeiro-Neto's
  'Modern Information Retrieval' at:
    /home/alexander/RI/ExamenSupletorio/corpus/baeza-yates-modern-ir.pdf
  The bibliographic metadata is already registered in
  src/ir_rag/corpus.py. If your edition has different authors
  or year, edit that registry accordingly.


In [4]:
from ir_rag.corpus import discover_sources, CORPUS_REGISTRY

print('Bibliografía registrada en el sistema:')
for filename, meta in CORPUS_REGISTRY.items():
    print(f'  - {meta["doc_id"]:>22s}  [{meta["kind"]:>7s}]  {filename}')

sources = discover_sources(settings.corpus_dir, settings.articles_dir)
print(f'\nFuentes encontradas en disco: {len(sources)}')
for s in sources:
    print(f'  - {s.doc_id:>22s}  {Path(s.file_path).relative_to(PROJECT_ROOT)}')

Bibliografía registrada en el sistema:
  -       baeza-yates-1999  [   book]  baeza-yates-modern-ir.pdf
  -           manning-2009  [   book]  manning-introduction-ir.pdf
  -   jurafsky-martin-2026  [   book]  jurafsky-slp3.pdf
  - robertson-zaragoza-2009  [article]  robertson-bm25-perspective.pdf
  -    karpukhin-etal-2020  [article]  karpukhin-dpr-2020.pdf
  -      nogueira-cho-2019  [article]  nogueira-monobert-2019.pdf

Fuentes encontradas en disco: 5
  -   jurafsky-martin-2026  corpus/jurafsky-slp3.pdf
  -           manning-2009  corpus/manning-introduction-ir.pdf
  -    karpukhin-etal-2020  corpus/articles/karpukhin-dpr-2020.pdf
  -      nogueira-cho-2019  corpus/articles/nogueira-monobert-2019.pdf
  - robertson-zaragoza-2009  corpus/articles/robertson-bm25-perspective.pdf


## C. Procesamiento del corpus

Cada PDF pasa por:

1. Extracción de texto página por página (PyMuPDF).
2. Detección y eliminación de encabezados/pies repetidos (umbral relativo al total de páginas).
3. Identificación de capítulos y secciones por número + tamaño de fuente + negrita.
4. División en *chunks* de `CHUNK_SIZE` caracteres con solapamiento.
5. Asignación de ID único `{doc_id}:p{page}:c{idx:04d}-{slug}`.

In [5]:
from ir_rag.corpus import extract_pages

# Vista rápida del primer libro obligatorio que encontremos en disco.
manning = next(s for s in sources if s.doc_id == 'manning-2009')
pages = extract_pages(manning)
print(f'"{manning.title}" → {len(pages)} páginas tras limpieza')

# Mostrar la primera página con encabezado/sección detectados
for i, p in enumerate(pages[:60]):
    if p.chapter:
        print(f'  p.{p.page_number}: chapter = {p.chapter!r}')
        if p.section:
            print(f'           section = {p.section!r}')
        break

"An Introduction to Information Retrieval" → 562 páginas tras limpieza
  p.38: chapter = '1 Boolean retrieval'


In [6]:
from ir_rag.corpus import build_corpus

chunks, stats = build_corpus(
    sources,
    chunk_size=settings.chunk_size,
    chunk_overlap=settings.chunk_overlap,
    min_chars=settings.min_chunk_chars,
)

print(f'Total chunks: {len(chunks)}')
for doc_id, n in stats['per_source'].items():
    print(f'  {doc_id:>22s}: {n:>5d} chunks')
print(f'Fallos: {stats["failed"]}')

Total chunks: 3584
    jurafsky-martin-2026:  1977 chunks
            manning-2009:  1420 chunks
     karpukhin-etal-2020:    59 chunks
       nogueira-cho-2019:    15 chunks
  robertson-zaragoza-2009:   113 chunks
Fallos: []


In [7]:
# Inspección de un chunk real (debe incluir página y sección).
for c in chunks:
    if c.doc_id == 'manning-2009' and c.section:
        print(f'chunk_id    = {c.chunk_id}')
        print(f'doc_id      = {c.doc_id}')
        print(f'title       = {c.title}')
        print(f'authors     = {", ".join(c.authors)}')
        print(f'chapter     = {c.chapter}')
        print(f'section     = {c.section}')
        print(f'page_start  = {c.page_start}')
        print(f'page_end    = {c.page_end}')
        print(f'--- text ---')
        print(c.text[:600] + ('...' if len(c.text) > 600 else ''))
        break

chunk_id    = manning-2009:p40:c0063-1-1-an-example-information-ret
doc_id      = manning-2009
title       = An Introduction to Information Retrieval
authors     = Christopher D. Manning, Prabhakar Raghavan, Hinrich Schütze
chapter     = 1 Boolean retrieval
section     = 1.1 An example information retrieval problem
page_start  = 40
page_end    = 40
--- text ---
1.1
An example information retrieval problem
In this chapter we begin with a very simple example of an information
retrieval problem, and introduce the idea of a term-document matrix (Sec-
tion 1.1) and the central inverted index data structure (Section 1.2). We will
then examine the Boolean retrieval model and how Boolean queries are pro-
cessed (Sections 1.3 and 1.4).
1.1
An example information retrieval problem
A fat book which many people own is Shakespeare’s Collected Works. Sup-
pose you wanted to determine which plays of Shakespeare contain the words
Brutus AND Caesar AND NOT Calpurnia....


## D. Indexación en base vectorial

Embeddings con `sentence-transformers/all-MiniLM-L6-v2` (normalizados) y Chroma persistente con cosine similarity.

In [8]:
from ir_rag.indexing import build_index

def progress(done: int, total: int) -> None:
    print(f'  embedding {done:>6d}/{total:<6d} chunks', flush=True)

manifest = build_index(settings, progress=progress)
print('\nManifest:')
for key, value in manifest['stats'].items():
    print(f'  {key:>20s}: {value}')

/home/alexander/.local/lib/python3.12/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


Resetting collection 'ir_corpus'


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

  embedding    128/3584   chunks


  embedding    256/3584   chunks


...

  embedding   3200/3584   chunks


  embedding   3328/3584   chunks


  embedding   3456/3584   chunks


  embedding   3584/3584   chunks



Manifest:
             documents: 5
                chunks: 3584
            per_source: {'jurafsky-martin-2026': 1977, 'manning-2009': 1420, 'karpukhin-etal-2020': 59, 'nogueira-cho-2019': 15, 'robertson-zaragoza-2009': 113}
                failed: []
      collection_count: 3584
     embedding_seconds: 183.78


## E. Retrieval denso + re-ranking

Cada consulta:
1. Se codifica con el mismo modelo de embeddings.
2. Se buscan los `RETRIEVAL_CANDIDATES` (20) vecinos más cercanos.
3. Se reordenan con el cross-encoder `ms-marco-MiniLM-L-6-v2`.
4. Se eligen los `TOP_K` (5) más relevantes, con diversidad por documento.

In [9]:
from ir_rag.retriever import Retriever

retriever = Retriever(settings)
queries = [
    '¿Qué es BM25 y cómo funciona?',
    'How does dense passage retrieval differ from BM25?',
    'Explica la diferencia entre precisión y exhaustividad.',
    'What is PageRank and how is it used in web search?',
]

for q in queries:
    outcome = retriever.retrieve(q)
    print(f'\nQ: {q}')
    print(f'  insufficient = {outcome.insufficient}')
    for ev in outcome.evidence[:2]:
        print(f'  [{ev.evidence_id}] {ev.title[:45]:45s} sem={ev.semantic_score:.2f} rerank={ev.rerank_score:.2f}')
        print(f'      chunk={ev.chunk_id}')
        print(f'      p.{ev.page_start} {ev.chapter or "-"} / {ev.section or "-"}')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]


Q: ¿Qué es BM25 y cómo funciona?
  insufficient = True
  [E1] The Probabilistic Relevance Framework: BM25 a sem=0.36 rerank=0.04
      chunk=robertson-zaragoza-2009:p30:c0051-3-5-uses-of-bm25
      p.30 2 Development of the Basic Model / 3.5 Uses of BM25
  [E2] The Probabilistic Relevance Framework: BM25 a sem=0.27 rerank=0.03
      chunk=robertson-zaragoza-2009:p3:c0002-chunk
      p.3 - / -



Q: How does dense passage retrieval differ from BM25?
  insufficient = False
  [E1] Dense Passage Retrieval for Open-Domain Quest sem=0.61 rerank=1.00
      chunk=karpukhin-etal-2020:p7:c0028-chunk
      p.7 - / -
  [E2] Dense Passage Retrieval for Open-Domain Quest sem=0.61 rerank=1.00
      chunk=karpukhin-etal-2020:p5:c0021-chunk
      p.5 - / -



Q: Explica la diferencia entre precisión y exhaustividad.
  insufficient = True
  [E1] An Introduction to Information Retrieval      sem=0.49 rerank=0.00
      chunk=manning-2009:p210:c0484-8-7-results-snippets
      p.210 8 Evaluation in information / 8.7 Results snippets
  [E2] Speech and Language Processing                sem=0.40 rerank=0.00
      chunk=jurafsky-martin-2026:p285:c0850-12-6-2-automatic-evaluation
      p.285 - / 12.6.2 Automatic Evaluation



Q: What is PageRank and how is it used in web search?
  insufficient = False
  [E1] An Introduction to Information Retrieval      sem=0.64 rerank=1.00
      chunk=manning-2009:p506:c1222-21-2-pagerank
      p.506 21 Link analysis / 21.2 PageRank
  [E2] An Introduction to Information Retrieval      sem=0.56 rerank=0.99
      chunk=manning-2009:p508:c1226-21-2-pagerank
      p.508 21 Link analysis / 21.2 PageRank


## F. Generación de respuestas

El LLM recibe únicamente la evidencia del corpus y debe citar los identificadores `[E1]…[E5]`. Si la evidencia es insuficiente, devuelve `[INSUFFICIENT_CONTEXT]` y el servicio responde con un mensaje explícito sin invocar al LLM.

In [10]:
from ir_rag.rag import RAGPipeline, extract_question
from ir_rag.models import AnswerRequest

pipeline = RAGPipeline(settings)

# Markdown típico de un examen: encabezado, negrita y enlace.
md_q = (
    '## Pregunta\n\n'
    '¿Cuáles son los **componentes principales** del modelo BM25 y '
    'cómo se relacionan con [Information Retrieval](https://en.wikipedia.org/wiki/Information_retrieval)?'
)
print('Pregunta Markdown:')
print(md_q)
print('\nPregunta limpia (extract_question):')
print(extract_question(md_q))

Pregunta Markdown:
## Pregunta

¿Cuáles son los **componentes principales** del modelo BM25 y cómo se relacionan con [Information Retrieval](https://en.wikipedia.org/wiki/Information_retrieval)?

Pregunta limpia (extract_question):
Pregunta

¿Cuáles son los componentes principales del modelo BM25 y cómo se relacionan con Information Retrieval?


In [11]:
# Solo ejecuta si hay clave del LLM configurada (de lo contrario se lanza 503).
if settings.llm_api_key:
    response = pipeline.answer(AnswerRequest(question=md_q))
    print('Pregunta :', response.question)
    print('\nRespuesta :')
    print(response.answer)
    print('\nReferencias :')
    for ref in response.references:
        print(' ', ref)
    print(f'\nRetrieval_ms = {response.retrieval_ms:.1f}, Generation_ms = {response.generation_ms:.1f}')
    print(f'Insufficient = {response.insufficient}')
else:
    print('LLM_API_KEY no configurada — define la clave antes de generar respuestas.')

LLM_API_KEY no configurada — define la clave antes de generar respuestas.


## G. Servicio web (FastAPI)

El servicio expone `POST /answer` y mantiene compatibilidad con Postman/curl. Arranca con `uvicorn ir_rag.api:app --host 0.0.0.0 --port 7860`.

In [12]:
from fastapi.testclient import TestClient
from ir_rag.api import create_app

app = create_app(settings)
client = TestClient(app)

print('GET /')
r = client.get('/')
print(f'  status={r.status_code} name={r.json()["name"]}')

print('GET /health')
r = client.get('/health')
print(f'  status={r.status_code} chunks={r.json().get("chunks")}')

print('POST /answer (pregunta vacía → 400)')
r = client.post('/answer', json={'question': ''})
print(f'  status={r.status_code} detail={r.json()["detail"]}')

print('POST /answer (markdown sin pregunta → 422)')
r = client.post('/answer', json={'question': '```\ncode\n```'})
print(f'  status={r.status_code} detail={r.json()["detail"]}')

print('GET /ruta-inexistente → 404')
r = client.get('/ruta-inexistente')
print(f'  status={r.status_code} detail={r.json()["detail"]}')

GET /
  status=200 name=ir-rag-service
GET /health
  status=200 chunks=3584
POST /answer (pregunta vacía → 400)
  status=400 detail=The 'question' field is required and cannot be empty.
POST /answer (markdown sin pregunta → 422)
  status=422 detail=The request is valid but no question could be extracted from the supplied Markdown content.
GET /ruta-inexistente → 404
  status=404 detail=Endpoint 'GET /ruta-inexistente' was not found.


## H. Despliegue y ejemplos de consumo

### H.1 URL del servicio

Tras desplegar en Hugging Face Spaces, Render u otra plataforma con HTTPS automático, pega aquí la URL pública.

**URL HTTPS**: `https://<usuario>-<space>.hf.space` *(reemplaza)*

### H.2 Consumir el servicio con curl

```bash
curl -X POST https://<HOST>/answer \
     -H 'Content-Type: application/json' \
     -d '{"question": "## ¿Qué es BM25?\n\nExplica el algoritmo y sus variantes."}'
```

### H.3 Consumir el servicio con Postman

1. Crea una nueva *request* `POST` apuntando a `{{base_url}}/answer`.
2. En *Body → raw → JSON*: `{"question": "## ¿Qué es BM25?"}`.
3. Pulsa *Send* y revisa la pestaña *Body*.

### H.4 Consumir el servicio desde Python

Ejecuta la celda siguiente reemplazando `<HOST>` por tu URL.

In [13]:
import json
import urllib.request

HOST = 'prone-snowiness-reviving.ngrok-free.dev'   # p. ej. 'alexandor31-ir-rag.hf.space'
QUESTION = '## ¿Cómo funciona el algoritmo BM25?\n\nExplica el modelo.'

if HOST.startswith('<'):
    print('Edita la variable HOST antes de ejecutar esta celda.')
else:
    req = urllib.request.Request(
        f'https://' + HOST + '/answer',
        data=json.dumps({'question': QUESTION}).encode('utf-8'),
        headers={'Content-Type': 'application/json'},
        method='POST',
    )
    with urllib.request.urlopen(req, timeout=60) as resp:
        body = json.loads(resp.read().decode('utf-8'))
    print(f'HTTP {resp.status}')
    print(json.dumps(body, indent=2, ensure_ascii=False)[:2000])

HTTP 200
{
  "question": "¿Cómo funciona el algoritmo BM25?\n\nExplica el modelo.",
  "answer": "No se proporciona información detallada sobre cómo funciona el algoritmo BM25 en los textos suministrados [E1] y [E2]. Sin embargo, se menciona que el algoritmo BM25 es parte del Marco de Relevancia Probabilística (PRF) [E2], y que ha llevado al desarrollo de nuevos modelos de recuperación de documentos, incluyendo BM25F. Para una explicación detallada del modelo BM25, sería necesario consultar fuentes adicionales que profundicen en su funcionamiento interno y las suposiciones de modelado probabilístico detrás de él.",
  "references": [
    "- Stephen Robertson, Hugo Zaragoza (2009). *The Probabilistic Relevance Framework: BM25 and Beyond*. 2 Development of the Basic Model — p. 39.",
    "- Stephen Robertson, Hugo Zaragoza (2009). *The Probabilistic Relevance Framework: BM25 and Beyond*. p. 3."
  ],
  "evidence": [
    {
      "evidence_id": "E1",
      "rank": 1,
      "chunk_id": "roberts

## I. Códigos de estado HTTP

| Código | Significado |
|--------|-------------|
| 200 | Solicitud procesada correctamente. |
| 400 | Solicitud inválida o ausencia del campo `question`. |
| 404 | Endpoint o recurso no encontrado. |
| 422 | La solicitud es válida pero no fue posible identificar preguntas en el contenido recibido. |
| 500 | Error interno del servidor. |
| 503 | El modelo, la base vectorial o algún componente requerido no se encuentra disponible. |

